# Pipeline do zero: CNEFE bruto -> setor censitario + CEP + renda

Este notebook monta a base `setor censitario -> CEP -> renda` diretamente a partir das fontes brutas locais:

- CSVs brutos do CNEFE por UF
- malha de setores censitarios `BR_setores_CD2022.shp`
- CSV de renda por setor censitario

A saida principal fica no grao:

`1 linha = 1 par (setor censitario, CEP)`

A pipeline foi desenhada para nao carregar os 18 GB do CNEFE de uma vez. Ela le os CSVs por chunks, agrega os dados em SQLite e exporta a base final em CSV e Parquet.

## Saidas geradas

Por padrao, este notebook grava os arquivos em `saida_setor_cep_renda_do_zero/`:

- `setor_censitario_cep_renda_do_zero.csv`
- `setor_censitario_cep_renda_do_zero.parquet`
- `setor_cep_renda_do_zero_work.sqlite`
- `setor_censitario_cep_renda_do_zero_resumo.json`

Se voce rodar com filtros ou limites de teste, a saida refletira apenas esse recorte.

In [ ]:
from pathlib import Path
import json
import sqlite3
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

try:
    import geopandas as gpd
    GEOPANDAS_AVAILABLE = True
except Exception:
    gpd = None
    GEOPANDAS_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print({"GEOPANDAS_AVAILABLE": GEOPANDAS_AVAILABLE})

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "saida_cnefe_uf").exists() and (candidate / "BR_setores_CD2022").exists():
            return candidate
    return start


PROJECT_ROOT_OVERRIDE = None
# Exemplo para a amostra versionada neste repositorio:
# PROJECT_ROOT_OVERRIDE = Path("sample_data/pipeline_do_zero_amostra")

PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve() if PROJECT_ROOT_OVERRIDE else find_project_root(Path.cwd())
CNEFE_ROOT = PROJECT_ROOT / "saida_cnefe_uf" / "extraido" / "SEM_UF"
RENDA_CSV = PROJECT_ROOT / "Agregados_por_setores_renda_responsavel_BR_csv" / "Agregados_por_setores_renda_responsavel_BR.csv"
SHAPEFILE = PROJECT_ROOT / "BR_setores_CD2022" / "BR_setores_CD2022.shp"

OUTPUT_DIR = PROJECT_ROOT / "saida_setor_cep_renda_do_zero"
WORK_SQLITE = OUTPUT_DIR / "setor_cep_renda_do_zero_work.sqlite"
OUT_CSV = OUTPUT_DIR / "setor_censitario_cep_renda_do_zero.csv"
OUT_PARQUET = OUTPUT_DIR / "setor_censitario_cep_renda_do_zero.parquet"
OUT_SUMMARY = OUTPUT_DIR / "setor_censitario_cep_renda_do_zero_resumo.json"

# Filtros opcionais para rodadas menores.
# Use None para processar Brasil inteiro.
UF_FILTER = None
# Exemplos: ['SP'] | ['35'] | ['SP', 'RJ']
SETOR_FILTER = None
# Exemplos: ['350010505000017']

# Limites opcionais para teste. Use None para carga completa.
LIMIT_FILES = None
LIMIT_ROWS_PER_FILE = None

# Parametros de performance.
CHUNKSIZE = 250_000
SQL_CHUNKSIZE = 100_000

# Se True, recria o SQLite de trabalho do zero.
# Se False, reaproveita o SQLite existente e refaz apenas a exportacao final.
REBUILD_SQLITE = True

EXPORT_CSV = True
EXPORT_PARQUET = True

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CNEFE_ROOT: {CNEFE_ROOT}")
print(f"RENDA_CSV: {RENDA_CSV}")
print(f"SHAPEFILE: {SHAPEFILE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## Funcoes auxiliares da pipeline

In [ ]:
UF_POR_SIGLA = {
    "RO": "11", "AC": "12", "AM": "13", "RR": "14", "PA": "15", "AP": "16", "TO": "17",
    "MA": "21", "PI": "22", "CE": "23", "RN": "24", "PB": "25", "PE": "26", "AL": "27",
    "SE": "28", "BA": "29", "MG": "31", "ES": "32", "RJ": "33", "SP": "35", "PR": "41",
    "SC": "42", "RS": "43", "MS": "50", "MT": "51", "GO": "52", "DF": "53",
}
SIGLA_POR_UF = {codigo: sigla for sigla, codigo in UF_POR_SIGLA.items()}

REQUIRED_CNEFE_COLUMNS = [
    "CEP",
    "COD_SETOR",
    "COD_UF",
    "COD_MUNICIPIO",
    "NOM_TIPO_SEGLOGR",
    "NOM_TITULO_SEGLOGR",
    "NOM_SEGLOGR",
]

REQUIRED_RENDA_COLUMNS = ["CD_SETOR", "V06001", "V06002", "V06003", "V06004", "V06005"]

FINAL_COLUMNS = [
    "cd_setor",
    "cep",
    "cod_uf",
    "sigla_uf",
    "nm_uf",
    "cod_municipio",
    "nm_municipio",
    "area_km2",
    "cep_inicial_setor",
    "cep_final_setor",
    "faixa_cep_setor",
    "qtd_ceps_no_setor",
    "qtd_setores_no_cep",
    "qtd_enderecos_setor_cep",
    "qtd_logradouros_distintos_setor_cep",
    "total_enderecos_no_setor",
    "total_enderecos_no_cep",
    "pct_enderecos_do_setor_no_cep",
    "pct_enderecos_do_cep_no_setor",
    "tem_renda",
    "renda_v06001",
    "renda_v06002",
    "renda_v06003",
    "renda_v06004",
    "renda_v06005",
]


def log(message: str) -> None:
    print(message, flush=True)


def now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def normalize_uf_filters(values):
    if not values:
        return None
    normalized = set()
    invalid = []
    for value in values:
        token = str(value).strip().upper()
        if token in UF_POR_SIGLA:
            normalized.add(UF_POR_SIGLA[token])
        elif token in SIGLA_POR_UF:
            normalized.add(token)
        else:
            invalid.append(value)
    if invalid:
        raise ValueError(f"UF(s) invalidas: {invalid}")
    return sorted(normalized) or None


def normalize_setor_filters(values):
    if not values:
        return None
    normalized = []
    invalid = []
    for value in values:
        token = str(value).strip()
        if len(token) == 15 and token.isdigit():
            normalized.append(token)
        else:
            invalid.append(value)
    if invalid:
        raise ValueError(f"CD_SETOR invalidos: {invalid}")
    return sorted(set(normalized)) or None


def infer_uf_code_from_filename(path: Path):
    prefix = path.stem.split("_", 1)[0]
    return prefix if prefix in SIGLA_POR_UF else None


def collect_cnefe_files(root: Path, uf_codes=None, limit_files=None):
    if not root.exists():
        raise FileNotFoundError(f"Pasta do CNEFE nao encontrada: {root}")
    files = sorted(path for path in root.rglob("*.csv") if path.is_file())
    if uf_codes:
        files = [path for path in files if infer_uf_code_from_filename(path) in set(uf_codes)]
    if limit_files is not None:
        files = files[:limit_files]
    if not files:
        raise ValueError("Nenhum CSV do CNEFE foi selecionado.")
    return files


def validate_columns(columns, required, source_name):
    missing = [column for column in required if column not in columns]
    if missing:
        raise ValueError(f"Colunas ausentes em {source_name}: {missing}")


def parse_br_number_series(series: pd.Series) -> pd.Series:
    normalized = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(normalized, errors="coerce")


def open_sqlite(path: Path) -> sqlite3.Connection:
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA synchronous = NORMAL")
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA cache_size = -200000")
    conn.execute("PRAGMA mmap_size = 30000000000")
    return conn


def reset_tables(conn: sqlite3.Connection) -> None:
    conn.executescript(
        """
        DROP TABLE IF EXISTS pares_logradouro;
        DROP TABLE IF EXISTS renda;
        DROP TABLE IF EXISTS setores;

        CREATE TABLE pares_logradouro (
            cd_setor TEXT NOT NULL,
            cep TEXT NOT NULL,
            logradouro TEXT NOT NULL,
            cod_uf TEXT NOT NULL,
            cod_municipio TEXT NOT NULL,
            qtd_enderecos INTEGER NOT NULL,
            PRIMARY KEY (cd_setor, cep, logradouro, cod_uf, cod_municipio)
        ) WITHOUT ROWID;

        CREATE TABLE renda (
            cd_setor TEXT PRIMARY KEY,
            renda_v06001 REAL,
            renda_v06002 REAL,
            renda_v06003 REAL,
            renda_v06004 REAL,
            renda_v06005 REAL
        ) WITHOUT ROWID;

        CREATE TABLE setores (
            cd_setor TEXT PRIMARY KEY,
            cod_uf_setor TEXT,
            nm_uf TEXT,
            cod_municipio_setor TEXT,
            nm_municipio TEXT,
            area_km2 REAL
        ) WITHOUT ROWID;
        """
    )
    conn.commit()


def load_renda_table(conn: sqlite3.Connection, renda_csv: Path, setor_filters=None) -> int:
    if not renda_csv.exists():
        raise FileNotFoundError(f"CSV de renda nao encontrado: {renda_csv}")
    log(f"[renda] Carregando {renda_csv}")
    renda_df = pd.read_csv(renda_csv, sep=";", dtype=str, keep_default_na=False, na_filter=False)
    validate_columns(renda_df.columns, REQUIRED_RENDA_COLUMNS, renda_csv.name)
    prepared = pd.DataFrame(
        {
            "cd_setor": renda_df["CD_SETOR"].astype(str).str.strip(),
            "renda_v06001": parse_br_number_series(renda_df["V06001"]),
            "renda_v06002": parse_br_number_series(renda_df["V06002"]),
            "renda_v06003": parse_br_number_series(renda_df["V06003"]),
            "renda_v06004": parse_br_number_series(renda_df["V06004"]),
            "renda_v06005": parse_br_number_series(renda_df["V06005"]),
        }
    )
    prepared = prepared.loc[prepared["cd_setor"].str.len() == 15].drop_duplicates("cd_setor", keep="last")
    if setor_filters:
        prepared = prepared.loc[prepared["cd_setor"].isin(set(setor_filters))].copy()
    prepared.to_sql("renda", conn, if_exists="append", index=False)
    conn.commit()
    log(f"[renda] Setores com renda: {len(prepared):,}")
    return int(len(prepared))


def load_setores_table(conn: sqlite3.Connection, shapefile: Path, setor_filters=None) -> int:
    if not GEOPANDAS_AVAILABLE:
        raise RuntimeError("geopandas nao esta disponivel; instale geopandas para ler o shapefile.")
    if not shapefile.exists():
        raise FileNotFoundError(f"Shapefile nao encontrado: {shapefile}")
    log(f"[setores] Carregando atributos do shapefile {shapefile}")
    setores_df = gpd.read_file(
        shapefile,
        columns=["CD_SETOR", "CD_UF", "NM_UF", "CD_MUN", "NM_MUN", "AREA_KM2"],
        ignore_geometry=True,
    )
    prepared = pd.DataFrame(
        {
            "cd_setor": setores_df["CD_SETOR"].fillna("").astype(str).str.strip(),
            "cod_uf_setor": setores_df["CD_UF"].fillna("").astype(str).str.strip().str.zfill(2),
            "nm_uf": setores_df["NM_UF"].fillna("").astype(str).str.strip(),
            "cod_municipio_setor": setores_df["CD_MUN"].fillna("").astype(str).str.strip().str.zfill(7),
            "nm_municipio": setores_df["NM_MUN"].fillna("").astype(str).str.strip(),
            "area_km2": pd.to_numeric(setores_df["AREA_KM2"], errors="coerce"),
        }
    )
    prepared = prepared.loc[prepared["cd_setor"].str.len() == 15].drop_duplicates("cd_setor", keep="last")
    if setor_filters:
        prepared = prepared.loc[prepared["cd_setor"].isin(set(setor_filters))].copy()
    prepared.to_sql("setores", conn, if_exists="append", index=False)
    conn.commit()
    log(f"[setores] Setores carregados: {len(prepared):,}")
    return int(len(prepared))

In [ ]:
def normalize_cnefe_chunk(chunk: pd.DataFrame, setor_filters=None):
    working = chunk.copy()
    for column in REQUIRED_CNEFE_COLUMNS:
        working[column] = working[column].fillna("").astype(str)

    cep_digits = working["CEP"].str.replace(r"\D+", "", regex=True)
    cep = cep_digits.where(cep_digits.eq(""), cep_digits.str.zfill(8))

    logradouro = (
        working["NOM_TIPO_SEGLOGR"].str.strip()
        + " "
        + working["NOM_TITULO_SEGLOGR"].str.strip()
        + " "
        + working["NOM_SEGLOGR"].str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    normalized = pd.DataFrame(
        {
            "cd_setor": working["COD_SETOR"].str.strip().str.slice(0, 15),
            "cep": cep,
            "logradouro": logradouro,
            "cod_uf": working["COD_UF"].str.strip().str.zfill(2),
            "cod_municipio": working["COD_MUNICIPIO"].str.strip().str.zfill(7),
        }
    )

    setor_filter_set = set(setor_filters or [])

    valid_mask = (
        normalized["cd_setor"].str.len().eq(15)
        & normalized["cep"].ne("")
        & normalized["logradouro"].ne("")
        & normalized["cod_uf"].ne("")
        & normalized["cod_municipio"].ne("")
    )
    if setor_filter_set:
        valid_mask = valid_mask & normalized["cd_setor"].isin(setor_filter_set)
    skipped_rows = int((~valid_mask).sum())

    grouped = (
        normalized.loc[valid_mask]
        .groupby(["cd_setor", "cep", "logradouro", "cod_uf", "cod_municipio"], sort=False, dropna=False)
        .size()
        .reset_index(name="qtd_enderecos")
    )
    return grouped, skipped_rows


def upsert_pares_logradouro(conn: sqlite3.Connection, grouped: pd.DataFrame) -> None:
    if grouped.empty:
        return
    conn.executemany(
        """
        INSERT INTO pares_logradouro (
            cd_setor, cep, logradouro, cod_uf, cod_municipio, qtd_enderecos
        )
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT (cd_setor, cep, logradouro, cod_uf, cod_municipio)
        DO UPDATE SET qtd_enderecos = pares_logradouro.qtd_enderecos + excluded.qtd_enderecos
        """,
        grouped.itertuples(index=False, name=None),
    )


def ingest_cnefe_files(conn: sqlite3.Connection, files, chunksize: int, limit_rows_per_file=None, setor_filters=None):
    stats = {
        "files_processed": 0,
        "chunks_processed": 0,
        "raw_rows_read": 0,
        "raw_rows_skipped": 0,
        "grouped_rows_upserted": 0,
    }

    for file_index, csv_path in enumerate(files, start=1):
        log(f"[cnefe] Arquivo {file_index}/{len(files)}: {csv_path.name}")
        file_rows_read = 0
        reader = pd.read_csv(
            csv_path,
            sep=";",
            dtype=str,
            usecols=REQUIRED_CNEFE_COLUMNS,
            keep_default_na=False,
            na_filter=False,
            chunksize=chunksize,
        )

        seen_columns = False
        for chunk_index, chunk in enumerate(reader, start=1):
            if not seen_columns:
                validate_columns(chunk.columns, REQUIRED_CNEFE_COLUMNS, csv_path.name)
                seen_columns = True

            if limit_rows_per_file is not None:
                remaining = limit_rows_per_file - file_rows_read
                if remaining <= 0:
                    break
                if len(chunk) > remaining:
                    chunk = chunk.iloc[:remaining].copy()

            raw_rows = int(len(chunk))
            if raw_rows == 0:
                continue

            grouped, skipped_rows = normalize_cnefe_chunk(chunk, setor_filters=setor_filters)
            with conn:
                upsert_pares_logradouro(conn, grouped)

            file_rows_read += raw_rows
            stats["chunks_processed"] += 1
            stats["raw_rows_read"] += raw_rows
            stats["raw_rows_skipped"] += skipped_rows
            stats["grouped_rows_upserted"] += int(len(grouped))

            log(
                f"[cnefe] {csv_path.name} chunk {chunk_index}: "
                f"linhas={raw_rows:,} validas={raw_rows - skipped_rows:,} grupos={len(grouped):,}"
            )

            if limit_rows_per_file is not None and file_rows_read >= limit_rows_per_file:
                break

        stats["files_processed"] += 1
        log(f"[cnefe] Finalizado {csv_path.name}: linhas lidas={file_rows_read:,}")

    return stats

In [ ]:
def create_indices(conn: sqlite3.Connection) -> None:
    log("[sqlite] Criando indices")
    conn.executescript(
        """
        CREATE INDEX IF NOT EXISTS idx_pares_cd_setor ON pares_logradouro (cd_setor);
        CREATE INDEX IF NOT EXISTS idx_pares_cep ON pares_logradouro (cep);
        CREATE INDEX IF NOT EXISTS idx_pares_cod_mun ON pares_logradouro (cod_municipio);
        CREATE INDEX IF NOT EXISTS idx_renda_cd_setor ON renda (cd_setor);
        CREATE INDEX IF NOT EXISTS idx_setores_cd_setor ON setores (cd_setor);
        """
    )
    conn.commit()


def final_query() -> str:
    return """
        WITH setor_cep AS (
            SELECT
                cd_setor,
                cep,
                MIN(cod_uf) AS cod_uf,
                MIN(cod_municipio) AS cod_municipio,
                SUM(qtd_enderecos) AS qtd_enderecos_setor_cep,
                COUNT(*) AS qtd_logradouros_distintos_setor_cep
            FROM pares_logradouro
            GROUP BY cd_setor, cep
        ),
        setor_stats AS (
            SELECT
                cd_setor,
                MIN(cep) AS cep_inicial_setor,
                MAX(cep) AS cep_final_setor,
                COUNT(*) AS qtd_ceps_no_setor,
                SUM(qtd_enderecos_setor_cep) AS total_enderecos_no_setor
            FROM setor_cep
            GROUP BY cd_setor
        ),
        cep_stats AS (
            SELECT
                cep,
                COUNT(*) AS qtd_setores_no_cep,
                SUM(qtd_enderecos_setor_cep) AS total_enderecos_no_cep
            FROM setor_cep
            GROUP BY cep
        )
        SELECT
            sc.cd_setor AS cd_setor,
            sc.cep AS cep,
            COALESCE(s.cod_uf_setor, sc.cod_uf) AS cod_uf,
            CASE COALESCE(s.cod_uf_setor, sc.cod_uf)
                WHEN '11' THEN 'RO' WHEN '12' THEN 'AC' WHEN '13' THEN 'AM' WHEN '14' THEN 'RR'
                WHEN '15' THEN 'PA' WHEN '16' THEN 'AP' WHEN '17' THEN 'TO' WHEN '21' THEN 'MA'
                WHEN '22' THEN 'PI' WHEN '23' THEN 'CE' WHEN '24' THEN 'RN' WHEN '25' THEN 'PB'
                WHEN '26' THEN 'PE' WHEN '27' THEN 'AL' WHEN '28' THEN 'SE' WHEN '29' THEN 'BA'
                WHEN '31' THEN 'MG' WHEN '32' THEN 'ES' WHEN '33' THEN 'RJ' WHEN '35' THEN 'SP'
                WHEN '41' THEN 'PR' WHEN '42' THEN 'SC' WHEN '43' THEN 'RS' WHEN '50' THEN 'MS'
                WHEN '51' THEN 'MT' WHEN '52' THEN 'GO' WHEN '53' THEN 'DF'
                ELSE NULL
            END AS sigla_uf,
            s.nm_uf AS nm_uf,
            COALESCE(s.cod_municipio_setor, sc.cod_municipio) AS cod_municipio,
            s.nm_municipio AS nm_municipio,
            s.area_km2 AS area_km2,
            ss.cep_inicial_setor AS cep_inicial_setor,
            ss.cep_final_setor AS cep_final_setor,
            CASE
                WHEN ss.cep_inicial_setor = ss.cep_final_setor THEN ss.cep_inicial_setor
                ELSE ss.cep_inicial_setor || ' - ' || ss.cep_final_setor
            END AS faixa_cep_setor,
            ss.qtd_ceps_no_setor AS qtd_ceps_no_setor,
            cs.qtd_setores_no_cep AS qtd_setores_no_cep,
            sc.qtd_enderecos_setor_cep AS qtd_enderecos_setor_cep,
            sc.qtd_logradouros_distintos_setor_cep AS qtd_logradouros_distintos_setor_cep,
            ss.total_enderecos_no_setor AS total_enderecos_no_setor,
            cs.total_enderecos_no_cep AS total_enderecos_no_cep,
            ROUND(CAST(sc.qtd_enderecos_setor_cep AS REAL) / NULLIF(ss.total_enderecos_no_setor, 0), 8)
                AS pct_enderecos_do_setor_no_cep,
            ROUND(CAST(sc.qtd_enderecos_setor_cep AS REAL) / NULLIF(cs.total_enderecos_no_cep, 0), 8)
                AS pct_enderecos_do_cep_no_setor,
            CASE WHEN r.cd_setor IS NOT NULL THEN 1 ELSE 0 END AS tem_renda,
            r.renda_v06001 AS renda_v06001,
            r.renda_v06002 AS renda_v06002,
            r.renda_v06003 AS renda_v06003,
            r.renda_v06004 AS renda_v06004,
            r.renda_v06005 AS renda_v06005
        FROM setor_cep sc
        INNER JOIN setor_stats ss ON ss.cd_setor = sc.cd_setor
        INNER JOIN cep_stats cs ON cs.cep = sc.cep
        LEFT JOIN renda r ON r.cd_setor = sc.cd_setor
        LEFT JOIN setores s ON s.cd_setor = sc.cd_setor
        ORDER BY sc.cd_setor, sc.qtd_enderecos_setor_cep DESC, sc.cep
    """


def coerce_final_chunk_types(chunk: pd.DataFrame) -> pd.DataFrame:
    out = chunk.copy()
    string_columns = [
        "cd_setor", "cep", "cod_uf", "sigla_uf", "nm_uf", "cod_municipio", "nm_municipio",
        "cep_inicial_setor", "cep_final_setor", "faixa_cep_setor",
    ]
    int_columns = [
        "qtd_ceps_no_setor", "qtd_setores_no_cep", "qtd_enderecos_setor_cep",
        "qtd_logradouros_distintos_setor_cep", "total_enderecos_no_setor",
        "total_enderecos_no_cep", "tem_renda",
    ]
    float_columns = [
        "area_km2", "pct_enderecos_do_setor_no_cep", "pct_enderecos_do_cep_no_setor",
        "renda_v06001", "renda_v06002", "renda_v06003", "renda_v06004", "renda_v06005",
    ]
    for column in string_columns:
        out[column] = out[column].fillna("").astype(str)
    for column in int_columns:
        out[column] = pd.to_numeric(out[column], errors="coerce").fillna(0).astype("int64")
    for column in float_columns:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    return out[FINAL_COLUMNS]


def remove_if_exists(path: Path) -> None:
    if path.exists():
        path.unlink()


def export_final_outputs(conn: sqlite3.Connection, out_csv: Path, out_parquet: Path, sql_chunksize: int):
    rows_written = 0
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    if EXPORT_CSV:
        remove_if_exists(out_csv)
    if EXPORT_PARQUET:
        remove_if_exists(out_parquet)

    parquet_writer = None
    header_written = False

    for chunk in pd.read_sql_query(final_query(), conn, chunksize=sql_chunksize):
        prepared = coerce_final_chunk_types(chunk)
        if EXPORT_CSV:
            prepared.to_csv(out_csv, sep=";", index=False, mode="a", header=not header_written, encoding="utf-8")
            header_written = True
        if EXPORT_PARQUET:
            table = pa.Table.from_pandas(prepared, preserve_index=False)
            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(out_parquet, table.schema)
            parquet_writer.write_table(table)
        rows_written += int(len(prepared))
        log(f"[export] Linhas exportadas acumuladas: {rows_written:,}")

    if parquet_writer is not None:
        parquet_writer.close()

    return rows_written

## Execucao da pipeline

A celula abaixo executa a carga do zero quando `REBUILD_SQLITE = True`. Para reaproveitar o SQLite de trabalho e somente refazer os exports, mude `REBUILD_SQLITE = False` na configuracao inicial.

In [ ]:
started = time.time()
summary = {
    "started_at": now_iso(),
    "project_root": str(PROJECT_ROOT),
    "cnefe_root": str(CNEFE_ROOT),
    "renda_csv": str(RENDA_CSV),
    "shapefile": str(SHAPEFILE),
    "output_dir": str(OUTPUT_DIR),
    "work_sqlite": str(WORK_SQLITE),
    "uf_filter": UF_FILTER,
    "setor_filter": SETOR_FILTER,
    "limit_files": LIMIT_FILES,
    "limit_rows_per_file": LIMIT_ROWS_PER_FILE,
    "chunksize": CHUNKSIZE,
    "sql_chunksize": SQL_CHUNKSIZE,
    "rebuild_sqlite": REBUILD_SQLITE,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
uf_codes = normalize_uf_filters(UF_FILTER)
setor_filters = normalize_setor_filters(SETOR_FILTER)
summary["setor_filter"] = setor_filters
cnefe_files = collect_cnefe_files(CNEFE_ROOT, uf_codes=uf_codes, limit_files=LIMIT_FILES)
summary["cnefe_files_selected"] = len(cnefe_files)
summary["cnefe_files"] = [str(path) for path in cnefe_files]

print(f"Arquivos CNEFE selecionados: {len(cnefe_files)}")
display(pd.DataFrame({"arquivo": [path.name for path in cnefe_files], "bytes": [path.stat().st_size for path in cnefe_files]}).head(30))

with open_sqlite(WORK_SQLITE) as conn:
    if REBUILD_SQLITE:
        reset_tables(conn)
        summary["renda_rows_loaded"] = load_renda_table(conn, RENDA_CSV, setor_filters=setor_filters)
        summary["setores_rows_loaded"] = load_setores_table(conn, SHAPEFILE, setor_filters=setor_filters)
        summary["cnefe_ingest"] = ingest_cnefe_files(
            conn,
            cnefe_files,
            chunksize=CHUNKSIZE,
            limit_rows_per_file=LIMIT_ROWS_PER_FILE,
            setor_filters=setor_filters,
        )
        create_indices(conn)
    else:
        log("[sqlite] Reaproveitando SQLite existente")

    summary["distinct_setor_cep_logradouro"] = int(conn.execute("SELECT COUNT(*) FROM pares_logradouro").fetchone()[0])
    summary["distinct_setores_cnefe"] = int(conn.execute("SELECT COUNT(DISTINCT cd_setor) FROM pares_logradouro").fetchone()[0])
    summary["distinct_ceps_cnefe"] = int(conn.execute("SELECT COUNT(DISTINCT cep) FROM pares_logradouro").fetchone()[0])
    if setor_filters:
        summary["setores_filtrados_encontrados"] = [
            row[0] for row in conn.execute("SELECT DISTINCT cd_setor FROM pares_logradouro ORDER BY cd_setor").fetchall()
        ]
        if not summary["setores_filtrados_encontrados"]:
            raise ValueError(
                "Nenhum cd_setor do SETOR_FILTER foi encontrado no recorte selecionado. "
                "Aumente LIMIT_ROWS_PER_FILE ou remova esse limite para esse teste."
            )

    summary["rows_exported"] = export_final_outputs(conn, OUT_CSV, OUT_PARQUET, SQL_CHUNKSIZE)

summary["finished_at"] = now_iso()
summary["elapsed_seconds"] = round(time.time() - started, 3)
summary["outputs"] = {
    "csv": str(OUT_CSV) if EXPORT_CSV else None,
    "parquet": str(OUT_PARQUET) if EXPORT_PARQUET else None,
    "sqlite": str(WORK_SQLITE),
    "summary_json": str(OUT_SUMMARY),
}

OUT_SUMMARY.write_text(json.dumps(summary, ensure_ascii=True, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=True, indent=2)[:4000])

## Validacao rapida da base gerada

In [ ]:
if OUT_PARQUET.exists():
    parquet_file = pq.ParquetFile(OUT_PARQUET)
    preview_df = parquet_file.read_row_group(0).to_pandas().head(10)
elif OUT_CSV.exists():
    preview_df = pd.read_csv(OUT_CSV, sep=";", nrows=10)
else:
    raise FileNotFoundError("Nenhuma saida final foi encontrada.")

display(preview_df)

In [ ]:
with open_sqlite(WORK_SQLITE) as conn:
    metricas_sql = pd.read_sql_query(
        """
        WITH setor_cep AS (
            SELECT cd_setor, cep, SUM(qtd_enderecos) AS qtd_enderecos_setor_cep
            FROM pares_logradouro
            GROUP BY cd_setor, cep
        )
        SELECT
            COUNT(*) AS pares_setor_cep,
            COUNT(DISTINCT cd_setor) AS setores_distintos,
            COUNT(DISTINCT cep) AS ceps_distintos,
            SUM(qtd_enderecos_setor_cep) AS total_enderecos,
            AVG(qtd_enderecos_setor_cep) AS media_enderecos_por_par_setor_cep
        FROM setor_cep
        """,
        conn,
    )

display(metricas_sql)

In [ ]:
with open_sqlite(WORK_SQLITE) as conn:
    top_setores = pd.read_sql_query(
        """
        WITH setor_cep AS (
            SELECT cd_setor, cep, SUM(qtd_enderecos) AS qtd_enderecos_setor_cep
            FROM pares_logradouro
            GROUP BY cd_setor, cep
        )
        SELECT
            sc.cd_setor,
            MIN(sc.cep) AS cep_inicial_setor,
            MAX(sc.cep) AS cep_final_setor,
            COUNT(*) AS qtd_ceps_no_setor,
            SUM(sc.qtd_enderecos_setor_cep) AS total_enderecos_no_setor,
            r.renda_v06004,
            r.renda_v06005
        FROM setor_cep sc
        LEFT JOIN renda r ON r.cd_setor = sc.cd_setor
        GROUP BY sc.cd_setor
        ORDER BY qtd_ceps_no_setor DESC, total_enderecos_no_setor DESC
        LIMIT 20
        """,
        conn,
    )

display(top_setores)

## Mapas usando o output final

As celulas abaixo usam o arquivo final `setor_censitario_cep_renda_do_zero.parquet` para criar tres visoes:

- mapa de renda por setor
- mapa de quantidade de CEPs por setor
- rotulos com `cd_setor`, `faixa_cep_setor` e renda

Como a malha nacional e grande, o mapa deve ser feito com um recorte. Se voce nao definir filtro, o notebook escolhe automaticamente o municipio com mais setores na base final carregada.

In [ ]:
# Filtros opcionais para o mapa.
# Use codigos IBGE: UF com 2 digitos e municipio com 7 digitos.
MAP_COD_UF = None
MAP_COD_MUNICIPIO = None

# Quantidade de setores rotulados no mapa da direita.
MAP_TOP_LABELS = 8

# Se o recorte ainda ficar grande, use um limite visual para plotagem.
# None plota todos os setores do recorte.
MAP_MAX_SETORES = None

print(
    {
        "MAP_COD_UF": MAP_COD_UF,
        "MAP_COD_MUNICIPIO": MAP_COD_MUNICIPIO,
        "MAP_TOP_LABELS": MAP_TOP_LABELS,
        "MAP_MAX_SETORES": MAP_MAX_SETORES,
    }
)

In [ ]:
if not GEOPANDAS_AVAILABLE:
    raise RuntimeError("geopandas nao esta disponivel; instale geopandas para gerar os mapas.")
if not SHAPEFILE.exists():
    raise FileNotFoundError(f"Shapefile nao encontrado: {SHAPEFILE}")
if not OUT_PARQUET.exists() and not OUT_CSV.exists():
    raise FileNotFoundError("Output final nao encontrado. Rode a pipeline antes de gerar os mapas.")

if OUT_PARQUET.exists():
    final_base = pd.read_parquet(OUT_PARQUET)
    map_source = OUT_PARQUET
else:
    final_base = pd.read_csv(OUT_CSV, sep=";")
    map_source = OUT_CSV

for column in ["cd_setor", "cep", "cod_uf", "sigla_uf", "cod_municipio", "faixa_cep_setor"]:
    final_base[column] = final_base[column].fillna("").astype(str).str.strip()

for column in ["qtd_ceps_no_setor", "total_enderecos_no_setor", "renda_v06004", "renda_v06005", "tem_renda"]:
    final_base[column] = pd.to_numeric(final_base[column], errors="coerce")

final_base["cod_uf"] = final_base["cod_uf"].str.zfill(2)
final_base["cod_municipio"] = final_base["cod_municipio"].str.zfill(7)

map_setores = (
    final_base.groupby(["cd_setor", "cod_uf", "sigla_uf", "cod_municipio"], as_index=False)
    .agg(
        nm_uf=("nm_uf", "first"),
        nm_municipio=("nm_municipio", "first"),
        area_km2=("area_km2", "first"),
        cep_inicial_setor=("cep_inicial_setor", "first"),
        cep_final_setor=("cep_final_setor", "first"),
        faixa_cep_setor=("faixa_cep_setor", "first"),
        qtd_ceps_no_setor=("qtd_ceps_no_setor", "first"),
        total_enderecos_no_setor=("total_enderecos_no_setor", "first"),
        tem_renda=("tem_renda", "max"),
        renda_v06004=("renda_v06004", "first"),
        renda_v06005=("renda_v06005", "first"),
    )
)

if MAP_COD_MUNICIPIO is not None:
    map_cod_municipio = str(MAP_COD_MUNICIPIO).zfill(7)
    map_setores = map_setores.loc[map_setores["cod_municipio"] == map_cod_municipio].copy()
    where_clause = f"CD_MUN = '{map_cod_municipio}'"
elif MAP_COD_UF is not None:
    map_cod_uf = str(MAP_COD_UF).zfill(2)
    map_setores = map_setores.loc[map_setores["cod_uf"] == map_cod_uf].copy()
    where_clause = f"CD_UF = '{map_cod_uf}'"
else:
    municipio_auto = (
        map_setores.groupby(["cod_municipio", "cod_uf", "nm_municipio"], as_index=False)["cd_setor"]
        .nunique()
        .sort_values(["cd_setor", "cod_municipio"], ascending=[False, True])
        .iloc[0]
    )
    map_cod_municipio = str(municipio_auto["cod_municipio"]).zfill(7)
    map_setores = map_setores.loc[map_setores["cod_municipio"] == map_cod_municipio].copy()
    where_clause = f"CD_MUN = '{map_cod_municipio}'"
    print(
        f"Recorte automatico para mapa: {municipio_auto['nm_municipio']} "
        f"({map_cod_municipio}), UF {municipio_auto['cod_uf']}"
    )

if MAP_MAX_SETORES is not None and len(map_setores) > MAP_MAX_SETORES:
    map_setores = (
        map_setores.sort_values(["qtd_ceps_no_setor", "total_enderecos_no_setor"], ascending=[False, False])
        .head(MAP_MAX_SETORES)
        .copy()
    )

if len(map_setores) == 0:
    raise ValueError("Nenhum setor encontrado para o recorte configurado do mapa.")

geo_setores = gpd.read_file(
    SHAPEFILE,
    where=where_clause,
    columns=["CD_SETOR", "CD_UF", "NM_UF", "CD_MUN", "NM_MUN", "geometry"],
)
geo_setores["CD_SETOR"] = geo_setores["CD_SETOR"].fillna("").astype(str).str.strip()
geo_setores["CD_UF"] = geo_setores["CD_UF"].fillna("").astype(str).str.zfill(2)
geo_setores["CD_MUN"] = geo_setores["CD_MUN"].fillna("").astype(str).str.zfill(7)

geo_plot = geo_setores.merge(map_setores, left_on="CD_SETOR", right_on="cd_setor", how="inner")
geo_plot = geo_plot.to_crs(4674)

print(f"Fonte do mapa: {map_source}")
print(f"Setores no recorte tabular: {len(map_setores):,}")
print(f"Setores com geometria no mapa: {len(geo_plot):,}")
display(
    geo_plot[
        [
            "cd_setor",
            "NM_UF",
            "NM_MUN",
            "faixa_cep_setor",
            "qtd_ceps_no_setor",
            "total_enderecos_no_setor",
            "renda_v06004",
        ]
    ]
    .sort_values(["qtd_ceps_no_setor", "total_enderecos_no_setor"], ascending=[False, False])
    .head(20)
)

In [ ]:
if len(geo_plot) == 0:
    raise ValueError("Nao ha geometrias para plotar no recorte atual.")

fig, axes = plt.subplots(1, 2, figsize=(19, 9))

# 1) Mapa de renda por setor.
geo_renda = geo_plot.dropna(subset=["renda_v06004"])
if len(geo_renda):
    geo_renda.plot(
        column="renda_v06004",
        cmap="YlOrRd",
        linewidth=0.08,
        edgecolor="black",
        legend=True,
        ax=axes[0],
    )
else:
    geo_plot.boundary.plot(ax=axes[0], color="black", linewidth=0.2)
    axes[0].text(0.5, 0.5, "Sem renda disponivel", ha="center", va="center", transform=axes[0].transAxes)

axes[0].set_title("Mapa de renda por setor censitario (renda_v06004)")
axes[0].set_axis_off()

# 2) Mapa de quantidade de CEPs por setor.
geo_plot.plot(
    column="qtd_ceps_no_setor",
    cmap="Blues",
    linewidth=0.08,
    edgecolor="black",
    legend=True,
    ax=axes[1],
)
axes[1].set_title("Mapa de quantidade de CEPs por setor")
axes[1].set_axis_off()

# 3) Rotulos com cd_setor, faixa_cep_setor e renda.
rotulos = geo_plot.sort_values(
    ["qtd_ceps_no_setor", "total_enderecos_no_setor", "cd_setor"],
    ascending=[False, False, True],
).head(MAP_TOP_LABELS)

for _, row in rotulos.iterrows():
    ponto = row.geometry.representative_point()
    renda_texto = "sem renda" if pd.isna(row["renda_v06004"]) else f"renda {row['renda_v06004']:,.2f}"
    label = f"{row['cd_setor']}\nCEP {row['faixa_cep_setor']}\n{renda_texto}"
    axes[1].annotate(
        label,
        xy=(ponto.x, ponto.y),
        xytext=(3, 3),
        textcoords="offset points",
        fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.75),
    )

plt.tight_layout()
plt.show()